In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_mpt import StockMPT, LinearModel, NaiveModel
from dataloader_builder_mpt import build_dataloaders
from setup import StockMPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor
from model_training_mpt import model_setup, train_model_cuda

from model_training_mpt import train_model_cuda, evaluate_model, evaluate_best_model, precision_recall_curve
from model_analysis_mpt import test_model, print_loss_analysis, process_losses, format_num

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
print(path_data_preprocessor)
dls, train_norms = build_dataloaders(path_data_preprocessor)

preprocessed_data/data_1min_2021_2026_2
Building DataLoaders...
Train dataset samples: 88,247
Train loader batches:  689
Batch size:            128


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 10

eval_bs = 1000

stockMPT, stockMPT_params, opt1, sca1, sch1 = model_setup(StockMPT, StockMPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

3415808
4608


NaiveModel()

In [5]:
model_train_losses, model_val_losses = train_model_cuda(stockMPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Continuing from previous checkpoint...
[] []


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 8:

Learning Rate: 4.00e-04



|███▎      | 33.3% (03:40) Evaluating model on validation data... (206/207) [1585/4755]:                  

Epoch 8:
Training Loss:
   (CE)   0.5787445902824402
   (ACC)  0.7702656984329224
   (PREC) tensor([0.4337, 0.8004, 0.4516], device='cuda:0')
   (REC)  tensor([0.1750, 0.9721, 0.1261], device='cuda:0')
   (F1)   tensor([0.2493, 0.8779, 0.1972], device='cuda:0')
Validation Loss:
   (CE)   0.5851213335990906
   (ACC)  0.7649735808372498
   (PREC) tensor([0.4277, 0.7988, 0.4370], device='cuda:0')
   (REC)  tensor([0.1901, 0.9700, 0.1252], device='cuda:0')
   (F1)   tensor([0.2632, 0.8761, 0.1946], device='cuda:0')

Best Validation: 0.5851213335990906
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



...

|████████  | 80.1% (01:49) Training LinearModel-v11-111-2-183... [12690/15850]:            

Epoch 8:
Training Loss:
   (CE)   0.6608613133430481
   (ACC)  0.7517192959785461
   (PREC) tensor([0.3133, 0.7814, 0.3808], device='cuda:0')
   (REC)  tensor([0.1418, 0.9687, 0.0295], device='cuda:0')
   (F1)   tensor([0.1953, 0.8650, 0.0547], device='cuda:0')
Validation Loss:
   (CE)   0.6653286218643188
   (ACC)  0.7477858662605286
   (PREC) tensor([0.3262, 0.7779, 0.3779], device='cuda:0')
   (REC)  tensor([0.1421, 0.9694, 0.0405], device='cuda:0')
   (F1)   tensor([0.1980, 0.8632, 0.0731], device='cuda:0')

Best Validation: 0.6653286218643188
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



|█████████ | 90.1% (02:05) Training LinearModel-v11-111-2-183... [14280/15850]:            

Epoch 9:
Training Loss:
   (CE)   0.6605883836746216
   (ACC)  0.7519270181655884
   (PREC) tensor([0.3155, 0.7809, 0.3754], device='cuda:0')
   (REC)  tensor([0.1389, 0.9694, 0.0297], device='cuda:0')
   (F1)   tensor([0.1929, 0.8650, 0.0550], device='cuda:0')
Validation Loss:
   (CE)   0.665005087852478
   (ACC)  0.7479858994483948
   (PREC) tensor([0.3293, 0.7773, 0.3741], device='cuda:0')
   (REC)  tensor([0.1401, 0.9702, 0.0393], device='cuda:0')
   (F1)   tensor([0.1966, 0.8631, 0.0712], device='cuda:0')

Best Validation: 0.665005087852478
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



Epoch 10:
Training Loss:
   (CE)   0.660181999206543
   (ACC)  0.7523296475410461
   (PREC) tensor([0.3162, 0.7805, 0.3824], device='cuda:0')
   (REC)  tensor([0.1359, 0.9704, 0.0298], device='cuda:0')
   (F1)   tensor([0.1901, 0.8652, 0.0552], device='cuda:0')
Validation Loss:
   (CE)   0.6648951768875122
   (ACC)  0.7482385635375977
   (PREC) tensor([0.3290, 0.7770, 0.3786], device='cuda:0')
   (REC)  tensor([0.1369, 0.9709, 0.0404], device='cuda:0')
   (F1)   tensor([0.1934, 0.8632, 0.0729], device='cuda:0')

Best Validation: 0.6648951768875122
----------------------------------------------------------------------------------------------------

Finished


## Model Analysis -------------------------

In [6]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
mpt_losses = evaluate_best_model(stockMPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
mpt_test_losses = test_model(dls["test"], stockMPT, device, eval_bs, analysis_pbar)


|███▎      | 32.5% (00:09) Evaluating model on training data... (15/689) [912/2808]:                      

[] []


|██████▍   | 64.2% (00:18) Evaluating model on training data... (11/689) [1804/2808]:    

[] []


|██████████| 100.0% (01:22) Evaluating model on testing data... (39/40) [2808/2808]:     

In [7]:
for key, features in [("CE", StockMPT_cfg["target_features"]),
                      ("ACC", StockMPT_cfg["target_features"]),
                      ("PREC", StockMPT_cfg["target_features"]),
                      ("REC", StockMPT_cfg["target_features"]),
                      ("F1", StockMPT_cfg["target_features"])]:
    print_loss_analysis(
        process_losses(mpt_losses + mpt_test_losses +
                       linear_losses + linear_test_losses +
                       naive_losses + naive_test_losses, key),
        [stockMPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
        [format_num(stockMPT_params), format_num(linearModel_params), "0"],
        features, key
    )


--------------------------------------------------------------------------------------------------------------

CE

--------------------------------------------------------------------------------------------------------------

StockMPT-v11-111-2-183: 3.4M
    Training:       0.5764
    Validation:     0.5835
    Testing:        0.5481
    
LinearModel-v11-111-2-183: 4.6K
    Training:       0.6602
    Validation:     0.6649
    Testing:        0.6282
    
NaiveModel-B1_183: 0
    Training:       0.8712
    Validation:     0.8750
    Testing:        0.8513
    

--------------------------------------------------------------------------------------------------------------



--------------------------------------------------------------------------------------------------------------

ACC

--------------------------------------------------------------------------------------------------------------

StockMPT-v11-111-2-183: 3.4M
    Training:       0.7711
    Validation:     0.7654
    

In [8]:
precision_recall_curve(dls["val"], stockMPT, device, cls=2)
print("--------------")
precision_recall_curve(dls["val"], stockMPT, device, cls=1)
print("--------------")
precision_recall_curve(dls["val"], stockMPT, device, cls=0)

0.10 | PREC 0.2312 | REC 0.8426 | N 4762511
0.15 | PREC 0.2687 | REC 0.7364 | N 3582311
0.20 | PREC 0.3039 | REC 0.6222 | N 2676424
0.25 | PREC 0.3383 | REC 0.5007 | N 1934346
0.30 | PREC 0.3740 | REC 0.3742 | N 1308090
0.35 | PREC 0.4111 | REC 0.2501 | N 795046
0.40 | PREC 0.4507 | REC 0.1436 | N 416519
0.45 | PREC 0.4967 | REC 0.0669 | N 176036
0.50 | PREC 0.5477 | REC 0.0291 | N 69362
0.55 | PREC 0.5936 | REC 0.0132 | N 28955
0.60 | PREC 0.6347 | REC 0.0056 | N 11478
0.65 | PREC 0.6689 | REC 0.0021 | N 4144
0.70 | PREC 0.7112 | REC 0.0007 | N 1198
0.75 | PREC 0.6935 | REC 0.0001 | N 248
0.80 | PREC 0.8421 | REC 0.0000 | N 19
0.85 | PREC 0.0000 | REC 0.0000 | N 0
0.90 | PREC 0.0000 | REC 0.0000 | N 0
0.95 | PREC 0.0000 | REC 0.0000 | N 0
--------------


|██████████| 100.0% (01:37) Evaluating model on testing data... (39/40) [2808/2808]: 

0.10 | PREC 0.7514 | REC 0.9995 | N 10251213
0.15 | PREC 0.7579 | REC 0.9978 | N 10145767
0.20 | PREC 0.7663 | REC 0.9944 | N 10001306
0.25 | PREC 0.7764 | REC 0.9889 | N 9815888
0.30 | PREC 0.7882 | REC 0.9803 | N 9584916
0.35 | PREC 0.8011 | REC 0.9685 | N 9316398
0.40 | PREC 0.8145 | REC 0.9532 | N 9018860
0.45 | PREC 0.8280 | REC 0.9344 | N 8696314
0.50 | PREC 0.8416 | REC 0.9118 | N 8349921
0.55 | PREC 0.8551 | REC 0.8851 | N 7977095
0.60 | PREC 0.8686 | REC 0.8537 | N 7574505
0.65 | PREC 0.8821 | REC 0.8166 | N 7134070
0.70 | PREC 0.8958 | REC 0.7722 | N 6643124
0.75 | PREC 0.9098 | REC 0.7184 | N 6084874
0.80 | PREC 0.9246 | REC 0.6516 | N 5431476
0.85 | PREC 0.9399 | REC 0.5658 | N 4638866
0.90 | PREC 0.9565 | REC 0.4490 | N 3617029
0.95 | PREC 0.9751 | REC 0.2758 | N 2179491
--------------
0.10 | PREC 0.2339 | REC 0.8435 | N 4664414
0.15 | PREC 0.2708 | REC 0.7433 | N 3550238
0.20 | PREC 0.3055 | REC 0.6364 | N 2694586
0.25 | PREC 0.3399 | REC 0.5219 | N 1986016
0.30 | PREC 0.

In [9]:
dls, train_norms = build_dataloaders("handpicked_data", False, drop_last = False)

Building DataLoaders...


In [10]:
test_losses = test_model(dls["test"], stockMPT, device, eval_bs)
print(test_losses)

KeyError: 'test'